In [ ]:
# ================================================================
# CELL 1: Mount Google Drive
# ================================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ================================================================
# CELL 2: Extract Dataset
# ================================================================

import zipfile
import os

zip_path = "/content/drive/MyDrive/Research/dataset/dataset_500/dataset.zip"
extract_path = "/content/data/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

!ls /content/data

DATASET_PATH = "/content/data/dataset"

dataset


In [ ]:
# ================================================================
# CELL 3: Install Libraries
# ================================================================

!pip install rasterio opencv-python cma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.3/328.3 kB 11.2 MB/s eta 0:00:00


In [ ]:
# ================================================================
# CELL 4: Import Everything
# ================================================================

import os
import gc
import pickle
import numpy as np
import rasterio
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

tf.get_logger().setLevel('ERROR')

In [ ]:
# ================================================================
# CELL 5: Image Loading Function
# ================================================================

IMG_SIZE = 64
NORMALIZE_DIV = 10000.0

def load_13band_image(tif_path, img_size=IMG_SIZE, normalize_div=NORMALIZE_DIV):
    """
    Reads exactly 13 bands (bands 1..13) from the GeoTIFF at tif_path,
    resizes to (img_size, img_size) and returns float32 array shaped (img_size, img_size, 13).
    """
    with rasterio.open(tif_path) as src:
        if src.count < 13:
            raise ValueError(f"{tif_path} has {src.count} bands, expected 13.")
        img = src.read(list(range(1, 14))).astype(np.float32)

    img = np.transpose(img, (1, 2, 0))
    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    img = img / normalize_div

    return img

In [ ]:
# ================================================================
# CELL 6: Load Dataset and Split
# ================================================================

X = []
y = []

class_names = sorted(os.listdir(DATASET_PATH))
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

print("Class mapping:")
for k, v in class_to_idx.items():
    print(k, "→", v)

for class_name in class_names:
    class_folder = os.path.join(DATASET_PATH, class_name)
    label = class_to_idx[class_name]

    for file in os.listdir(class_folder):
        if file.endswith(".tif"):
            img_path = os.path.join(class_folder, file)
            img = load_13band_image(img_path)
            X.append(img)
            y.append(label)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

print("X shape:", X.shape)
print("y shape:", y.shape)

# First split: train + temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Second split: validation + test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Free original arrays to save memory
del X, y, X_temp, y_temp
gc.collect()

# Convert to float16 to save memory (cuts RAM usage in half)
X_train = X_train.astype(np.float16)
X_val = X_val.astype(np.float16)
X_test = X_test.astype(np.float16)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Class mapping:
AnnualCrop → 0
Forest → 1
HerbaceousVegetation → 2
Highway → 3
Industrial → 4
Pasture → 5
PermanentCrop → 6
Residential → 7
River → 8
SeaLake → 9
X shape: (5000, 64, 64, 13)
y shape: (5000,)
Train: (3500, 64, 64, 13)
Validation: (750, 64, 64, 13)
Test: (750, 64, 64, 13)


In [ ]:
# ================================================================
# CELL 7: Model Building Functions
# ================================================================

activation_map = {
    0: "relu",
    1: "tanh"
}

def build_cnn_model(
    input_shape,
    num_classes,
    num_conv_layers,
    filters_list,
    kernel_size,
    activation,
    dropout_rate
):
    model = models.Sequential()
    model.add(layers.Input(shape=input_shape))

    for i in range(num_conv_layers):
        model.add(
            layers.Conv2D(
                filters=filters_list[i],
                kernel_size=(kernel_size, kernel_size),
                padding="same",
                activation=activation
            )
        )
        model.add(layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation=activation))
    model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(num_classes, activation="softmax"))

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

def print_architecture(
    num_conv_layers,
    filters_list,
    kernel_size,
    activation,
    dropout_rate
):
    print("\n==============================")
    print("Architecture tried:")
    print(f"Convolutional layers: {num_conv_layers}")
    print(f"Filters per layer: {filters_list}")
    print(f"Kernel size: {kernel_size}x{kernel_size}")
    print(f"Activation function: {activation}")
    print(f"Dropout rate: {dropout_rate:.2f}")
    print("==============================\n")

In [ ]:
# ================================================================
# CELL 8: CMA-ES Decode Function
# ================================================================

BOUNDS = {
    "num_conv_layers": (1, 3),
    "filters_1": (16, 64),
    "filters_2": (32, 128),
    "filters_3": (64, 256),
    "kernel_size": (3, 5),
    "activation_id": (0, 1),
    "dropout_rate": (0.0, 0.5)
}

def decode_architecture(x):
    num_conv_layers = int(np.clip(round(x[0]),
                                  *BOUNDS["num_conv_layers"]))

    filters_1 = int(np.clip(round(x[1]),
                            *BOUNDS["filters_1"]))

    filters_2 = int(np.clip(round(x[2]),
                            *BOUNDS["filters_2"]))

    filters_3 = int(np.clip(round(x[3]),
                            *BOUNDS["filters_3"]))

    kernel_size = int(np.clip(round(x[4]),
                              *BOUNDS["kernel_size"]))

    activation_id = int(np.clip(round(x[5]),
                                *BOUNDS["activation_id"]))
    activation = activation_map[activation_id]

    dropout_rate = float(np.clip(x[6],
                                 *BOUNDS["dropout_rate"]))

    filters_list = [filters_1]
    if num_conv_layers > 1:
        filters_list.append(filters_2)
    if num_conv_layers > 2:
        filters_list.append(filters_3)

    return {
        "num_conv_layers": num_conv_layers,
        "filters_list": filters_list,
        "kernel_size": kernel_size,
        "activation": activation,
        "dropout_rate": dropout_rate
    }

In [ ]:
# ================================================================
# CELL 9: Run Baseline CNN First
# ================================================================

print("=" * 50)
print("TRAINING BASELINE CNN")
print("=" * 50)

tf.keras.backend.clear_session()

baseline_model = build_cnn_model(
    input_shape=(64, 64, 13),
    num_classes=10,
    num_conv_layers=2,
    filters_list=[32, 64],
    kernel_size=3,
    activation="relu",
    dropout_rate=0.3
)

baseline_model.summary()

baseline_history = baseline_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=1
)

baseline_val_loss, baseline_val_acc = baseline_model.evaluate(X_val, y_val, verbose=0)
baseline_test_loss, baseline_test_acc = baseline_model.evaluate(X_test, y_test, verbose=0)

print(f"\n📊 Baseline Validation Accuracy: {baseline_val_acc:.4f}")
print(f"📊 Baseline Test Accuracy: {baseline_test_acc:.4f}")

# Save baseline results
SAVE_DIR = "/content/drive/MyDrive/Research/cma_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

baseline_model.save(os.path.join(SAVE_DIR, 'baseline_model.h5'))
print("💾 Baseline model saved!")

del baseline_model
gc.collect()

TRAINING BASELINE CNN


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │         3,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     2,097,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,120,842 (8.09 MB)

 Trainable params: 2,120,842 (8.09 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - accuracy: 0.4371 - loss: 1.5331 - val_accuracy: 0.5733 - val_loss: 1.1480
Epoch 2/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6394 - loss: 1.0220 - val_accuracy: 0.6827 - val_loss: 0.8790
Epoch 3/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7049 - loss: 0.8554 - val_accuracy: 0.7213 - val_loss: 0.7554
Epoch 4/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.7269 - loss: 0.7612 - val_accuracy: 0.7707 - val_loss: 0.6576
Epoch 5/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.7486 - loss: 0.7307 - val_accuracy: 0.8040 - val_loss: 0.5732
Epoch 6/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8000 - loss: 0.5892 - val_accuracy: 0.8013 - val_loss: 0.5341
Epoch 7/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8237 - loss: 0.5037 - val_accuracy: 0.7280 - val_loss: 0.7426
Epoch 8/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8240 - loss: 0.5087 - val_accu


📊 Baseline Validation Accuracy: 0.8467
📊 Baseline Test Accuracy: 0.8053
💾 Baseline model saved!


1423

In [ ]:
# ================================================================
# CELL 10: CMA-ES Search with Checkpoint/Resume
#
# THIS IS THE MAIN CELL. After it finishes 5 generations:
#   1. It will print "⚠️  NOW DO THIS:"
#   2. Go to Runtime → Restart runtime
#   3. Run ALL cells from Cell 1 to Cell 10 again
#   4. It will auto-resume from where it stopped
#   5. Repeat until all 15 generations are done
# ================================================================

import cma
import pickle

# --- SETTINGS (change these if you want) ---
TOTAL_GENERATIONS = 15
GENS_PER_CHUNK = 5       # generations before saving & restarting
POPSIZE = 20

x0 = [3, 32, 64, 128, 3, 0, 0.5]   # initial solution
sigma = 0.5                          # search radius

SAVE_DIR = "/content/drive/MyDrive/Research/cma_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
checkpoint_path = os.path.join(SAVE_DIR, "cma_checkpoint.pkl")

# --- LOAD OR CREATE CMA-ES STATE ---
if os.path.exists(checkpoint_path):
    print("📂 Loading checkpoint from Google Drive...")
    with open(checkpoint_path, 'rb') as f:
        checkpoint = pickle.load(f)

    es = checkpoint['es']
    best_val_accuracy = checkpoint['best_val_accuracy']
    best_architecture = checkpoint['best_architecture']
    completed_generations = checkpoint['completed_generations']
    all_results = checkpoint['all_results']
    print(f"✅ Resuming from generation {completed_generations + 1}")
    print(f"🏆 Best so far: {best_val_accuracy:.4f}")
else:
    print("🆕 Starting fresh CMA-ES search...")
    es = cma.CMAEvolutionStrategy(x0, sigma, {
        "popsize": POPSIZE,
        "verb_log": 0
    })
    best_val_accuracy = 0.0
    best_architecture = None
    completed_generations = 0
    all_results = []

# --- FITNESS FUNCTION ---
def fitness_function(x):
    global best_val_accuracy, best_architecture, all_results

    tf.keras.backend.clear_session()
    gc.collect()

    arch = decode_architecture(x)

    print_architecture(
        arch["num_conv_layers"],
        arch["filters_list"],
        arch["kernel_size"],
        arch["activation"],
        arch["dropout_rate"]
    )

    model = build_cnn_model(
        input_shape=(64, 64, 13),
        num_classes=10,
        num_conv_layers=arch["num_conv_layers"],
        filters_list=arch["filters_list"],
        kernel_size=arch["kernel_size"],
        activation=arch["activation"],
        dropout_rate=arch["dropout_rate"]
    )

    model.fit(
        X_train, y_train,
        epochs=10,
        batch_size=32,
        validation_data=(X_val, y_val),
        verbose=0
    )

    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    print(f"Validation Accuracy: {val_acc:.4f}")

    if val_acc > best_val_accuracy:
        best_val_accuracy = float(val_acc)
        best_architecture = {
            'num_conv_layers': arch['num_conv_layers'],
            'filters_list': arch['filters_list'],
            'kernel_size': arch['kernel_size'],
            'activation': arch['activation'],
            'dropout_rate': arch['dropout_rate']
        }
        try:
            model.save(os.path.join(SAVE_DIR, 'best_model.h5'))
            print('💾 Saved new best model!')
        except Exception as e:
            print('Could not save model:', e)

    fitness = 1.0 - val_acc

    all_results.append({
        'generation': completed_generations + 1,
        'architecture': arch,
        'val_acc': float(val_acc),
        'fitness': float(fitness)
    })

    del model
    gc.collect()

    return fitness

# --- RUN GENERATIONS ---
remaining = TOTAL_GENERATIONS - completed_generations
gens_to_run = min(GENS_PER_CHUNK, remaining)

if gens_to_run <= 0:
    print("\n🏁 ALL GENERATIONS ALREADY COMPLETED!")
    print(f"🏆 Best Validation Accuracy: {best_val_accuracy:.4f}")
    if best_architecture:
        for k, v in best_architecture.items():
            print(f"   {k}: {v}")

    # Load best model and evaluate on test set
    best_model_path = os.path.join(SAVE_DIR, 'best_model.h5')
    if os.path.exists(best_model_path):
        tf.keras.backend.clear_session()
        best_model = tf.keras.models.load_model(best_model_path)
        test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)
        print(f"\n📊 Final Test Accuracy: {test_acc:.4f}")
        del best_model
        gc.collect()

else:
    print(f"\n🚀 Running generations {completed_generations+1} to {completed_generations+gens_to_run} out of {TOTAL_GENERATIONS}")

    for g in range(gens_to_run):
        gen_num = completed_generations + g + 1
        print(f"\n{'='*50}")
        print(f"######## Generation {gen_num}/{TOTAL_GENERATIONS} ########")
        print(f"{'='*50}")

        solutions = es.ask()
        fitnesses = []

        for i, x in enumerate(solutions):
            print(f"\n--- Candidate {i+1}/{POPSIZE} (Gen {gen_num}) ---")
            f = fitness_function(x)
            fitnesses.append(f)

        es.tell(solutions, fitnesses)
        es.disp()

        # Print best so far after each generation
        print(f"\n🏆 Best after Gen {gen_num}: {best_val_accuracy:.4f}")

    completed_generations += gens_to_run

    # --- SAVE CHECKPOINT TO GOOGLE DRIVE ---
    checkpoint = {
        'es': es,
        'best_val_accuracy': best_val_accuracy,
        'best_architecture': best_architecture,
        'completed_generations': completed_generations,
        'all_results': all_results
    }

    with open(checkpoint_path, 'wb') as f:
        pickle.dump(checkpoint, f)

    print(f"\n{'='*50}")
    print(f"💾 CHECKPOINT SAVED!")
    print(f"   Completed: {completed_generations}/{TOTAL_GENERATIONS} generations")
    print(f"   Best accuracy: {best_val_accuracy:.4f}")
    if best_architecture:
        for k, v in best_architecture.items():
            print(f"   {k}: {v}")
    print(f"{'='*50}")

    if completed_generations < TOTAL_GENERATIONS:
        print(f"\n⚠️  NOW DO THIS:")
        print(f"   1. Go to Runtime → Restart runtime")
        print(f"   2. Run ALL cells again (Cell 1 through Cell 10)")
        print(f"   3. It will auto-resume from generation {completed_generations+1}")
        print(f"   4. Repeat until all {TOTAL_GENERATIONS} generations are done")
    else:
        print(f"\n🏁 ALL {TOTAL_GENERATIONS} GENERATIONS COMPLETE!")

        # Final evaluation
        best_model_path = os.path.join(SAVE_DIR, 'best_model.h5')
        if os.path.exists(best_model_path):
            tf.keras.backend.clear_session()
            best_model = tf.keras.models.load_model(best_model_path)
            test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)
            print(f"\n📊 Final Test Accuracy of Best CMA-ES Model: {test_acc:.4f}")
            del best_model
            gc.collect()

📂 Loading checkpoint from Google Drive...
✅ Resuming from generation 11
🏆 Best so far: 0.8720

🚀 Running generations 11 to 15 out of 15

######## Generation 11/15 ########

--- Candidate 1/20 (Gen 11) ---

Architecture tried:
Convolutional layers: 3
Filters per layer: [31, 62, 129]
Kernel size: 3x3
Activation function: tanh
Dropout rate: 0.00

Validation Accuracy: 0.8173

--- Candidate 2/20 (Gen 11) ---

Architecture tried:
Convolutional layers: 3
Filters per layer: [29, 63, 130]
Kernel size: 3x3
Activation function: tanh
Dropout rate: 0.00

Validation Accuracy: 0.7573

--- Candidate 3/20 (Gen 11) ---

Architecture tried:
Convolutional layers: 3
Filters per layer: [31, 63, 128]
Kernel size: 3x3
Activation function: tanh
Dropout rate: 0.00

Validation Accuracy: 0.8400

--- Candidate 4/20 (Gen 11) ---

Architecture tried:
Convolutional layers: 3
Filters per layer: [31, 64, 128]
Kernel size: 3x3
Activation function: tanh
Dropout rate: 0.00

Validation Accuracy: 0.8067

--- Candidate 5/20 


📊 Final Test Accuracy of Best CMA-ES Model: 0.8493


In [ ]:
# ================================================================
# CELL 11: Final Results Summary (run this AFTER all generations are done)
# ================================================================

# Only run this cell after all 15 generations are complete

checkpoint_path = "/content/drive/MyDrive/Research/cma_checkpoints/cma_checkpoint.pkl"

if os.path.exists(checkpoint_path):
    with open(checkpoint_path, 'rb') as f:
        checkpoint = pickle.load(f)

    print("=" * 50)
    print("FINAL RESULTS SUMMARY")
    print("=" * 50)
    print(f"Total generations completed: {checkpoint['completed_generations']}")
    print(f"Total architectures evaluated: {len(checkpoint['all_results'])}")
    print(f"\n🏆 Best Architecture:")
    for k, v in checkpoint['best_architecture'].items():
        print(f"   {k}: {v}")
    print(f"   Best Validation Accuracy: {checkpoint['best_val_accuracy']:.4f}")

    # Show accuracy progression per generation
    print(f"\n📈 Accuracy progression per generation:")
    for gen in range(1, checkpoint['completed_generations'] + 1):
        gen_results = [r for r in checkpoint['all_results'] if r['generation'] == gen]
        if gen_results:
            best_in_gen = max(r['val_acc'] for r in gen_results)
            avg_in_gen = np.mean([r['val_acc'] for r in gen_results])
            print(f"   Gen {gen:2d}: Best={best_in_gen:.4f}, Avg={avg_in_gen:.4f}")

    # Evaluate on test set
    best_model_path = "/content/drive/MyDrive/Research/cma_checkpoints/best_model.h5"
    if os.path.exists(best_model_path):
        tf.keras.backend.clear_session()
        best_model = tf.keras.models.load_model(best_model_path)
        test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)
        print(f"\n📊 CMA-ES Best Model Test Accuracy: {test_acc:.4f}")
        del best_model
        gc.collect()
else:
    print("No checkpoint found. Run Cell 10 first.")

FINAL RESULTS SUMMARY
Total generations completed: 15
Total architectures evaluated: 300

🏆 Best Architecture:
   num_conv_layers: 3
   filters_list: [31, 64, 128]
   kernel_size: 3
   activation: tanh
   dropout_rate: 0.0
   Best Validation Accuracy: 0.8720

📈 Accuracy progression per generation:
   Gen  1: Best=0.8720, Avg=0.8226
   Gen  6: Best=0.8627, Avg=0.8256
   Gen 11: Best=0.8653, Avg=0.8338



📊 CMA-ES Best Model Test Accuracy: 0.8493


In [ ]:
# --------------------  phase2   --------------------------------------